In [1]:
# ============================================================
# APL LOGISTICS ANALYTICS PROJECT
# Day 20 — Filters & Testing
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

# ============================================================
# 1. PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    r"D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations"
)

FEATURE_DATA_DIR = (
    PROJECT_ROOT / "data" / "feature_data"
)

ANALYSIS_DATA_DIR = (
    PROJECT_ROOT / "data" / "analysis_data"
)

APP_DIR = PROJECT_ROOT / "app"

DATA_FILE = (
    FEATURE_DATA_DIR / "APL_Day3_Feature_Engineered.csv"
)

MASTER_KPI_FILE = (
    ANALYSIS_DATA_DIR / "APL_Day15_Master_KPIs.csv"
)

DASHBOARD_INVENTORY_FILE = (
    ANALYSIS_DATA_DIR / "APL_Day15_Dashboard_Data_Inventory.csv"
)

# Create the output directory if necessary.

ANALYSIS_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Validate essential paths.
assert PROJECT_ROOT.exists(), (
    "Project root does not exist."
)

assert DATA_FILE.exists(), (
    "Feature-engineered dataset is missing."
)

assert APP_DIR.exists(), (
    "Dashboard app directory is missing."
)

print("Project Root:", PROJECT_ROOT)
print("Source Dataset:", DATA_FILE)
print("Analysis Directory:", ANALYSIS_DATA_DIR)
print("Dashboard Directory:", APP_DIR)

print("\nDay 20 project paths validated.")

Project Root: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations
Source Dataset: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\feature_data\APL_Day3_Feature_Engineered.csv
Analysis Directory: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data
Dashboard Directory: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\app

Day 20 project paths validated.


In [2]:
# ============================================================
# 2. LOAD SOURCE DATASET
# ============================================================

df = pd.read_csv(
    DATA_FILE,
    encoding="cp1252",
    low_memory=False
)

df.columns = (
    df.columns
    .astype(str)
    .str.strip()
)

print("Dataset Shape:", df.shape)

print("\nFirst Five Records:")
display(df.head())

print("\nAvailable Columns:")
print(df.columns.tolist())

print("\nDuplicate Rows:", df.duplicated().sum())

print(
    "\nTotal Missing Cells:",
    df.isna().sum().sum()
)

Dataset Shape: (180519, 54)

First Five Records:


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Delay_Severity,Delivery_Efficiency_Ratio,Revenue_Per_Unit,Profit_Margin,Discount_Level,Region_Market,Shipping_Mode_Rank,Is_Corporate_Customer,Order_Quantity_Category,Operational_Risk_Profile
0,DEBIT,6,4,159.69,472.45,Late delivery,1,9,Cardio Equipment,Brownsville,...,Major Delay,0.666667,99.99,0.319412,Low Discount,Pacific Asia - South Asia,4,0,Large,High Risk - Slower Shipping
1,DEBIT,4,4,48.71,167.96,Shipping on time,0,29,Shop By Sport,Littleton,...,On Time / Early,1.000000,39.99,0.243611,Medium Discount,LATAM - Central America,4,0,Large,Lower Risk
2,DEBIT,4,4,87.36,181.99,Shipping on time,0,48,Water Sports,Littleton,...,On Time / Early,1.000000,199.99,0.436822,Low Discount,LATAM - Central America,4,0,Small,Lower Risk
3,DEBIT,6,4,-41.89,175.99,Late delivery,1,48,Water Sports,Littleton,...,Major Delay,0.666667,199.99,-0.209460,Medium Discount,USCA - East of USA,4,0,Small,High Risk - Slower Shipping
4,DEBIT,6,4,10.00,40.00,Late delivery,1,24,Women's Apparel,Littleton,...,Major Delay,0.666667,50.00,0.200000,Medium Discount,USCA - East of USA,4,0,Small,High Risk - Slower Shipping



Available Columns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Product Name', 'Product Price', 'Shipping Mode', 'Shipping_Delay_Days', 'Is_Delayed', 'Is_On_Time', 'Early_Delivery_Days', 'Delay_Severity', 'Delivery_Efficiency_Ratio', 'Revenue_Per_Unit', 'Profit_Margin', 'Discount_Level', 'Region_Market', 'Shipping_Mode_Rank', 'Is_Corpor

In [3]:
# ============================================================
# 3. VALIDATE FILTER COLUMNS
# ============================================================

FILTER_COLUMNS = [
    "Shipping Mode",
    "Delivery Status",
    "Order Region",
    "Market",
    "Customer Segment",
    "Order Status",
    "Category Name",
    "Department Name"
]

filter_column_validation = pd.DataFrame({
    "Filter_Column": FILTER_COLUMNS,
    "Available": [
        col in df.columns
        for col in FILTER_COLUMNS
    ]
})

display(filter_column_validation)

missing_filter_columns = (
    filter_column_validation
    .loc[
        ~filter_column_validation["Available"],
        "Filter_Column"
    ]
    .tolist()
)

if missing_filter_columns:
    print(
        "Review missing filter columns:",
        missing_filter_columns
    )
else:
    print("All configured filter columns are available.")

,Filter_Column,Available
0,Shipping Mode,True
1,Delivery Status,True
2,Order Region,True
3,Market,True
4,Customer Segment,True
5,Order Status,True
6,Category Name,True
7,Department Name,True


All configured filter columns are available.


In [4]:
# ============================================================
# 4. REUSABLE FILTER FUNCTION
# ============================================================

def apply_dashboard_filters(
    data,
    filters=None
):
    """
    Apply multiple categorical filters to a DataFrame.

    Parameters
    ----------
    data : pandas.DataFrame
        Source or dashboard dataset.

    filters : dict
        Dictionary containing column names and selected values.

    Returns
    -------
    pandas.DataFrame
        Filtered copy of the source dataset.
    """

    if filters is None:
        filters = {}

    filtered_data = data.copy()

    for column, selected_values in filters.items():

        if column not in filtered_data.columns:
            raise KeyError(
                f"Filter column not found: {column}"
            )

        # None or an empty selection means no restriction.
        if selected_values is None:
            continue

        if isinstance(selected_values, str):
            selected_values = [selected_values]
        else:
            selected_values = list(selected_values)

        if len(selected_values) == 0:
            continue

        # Missing values are excluded from ordinary selections.
        filtered_data = filtered_data.loc[
            filtered_data[column].isin(
                selected_values
            )
        ]

    return filtered_data.copy()


print("Reusable dashboard filter function created.")

Reusable dashboard filter function created.


In [5]:
# ============================================================
# 5. BASIC FILTER TEST
# ============================================================

shipping_modes = sorted(
    df["Shipping Mode"]
    .dropna()
    .unique()
    .tolist()
)

print("Available Shipping Modes:")
print(shipping_modes)

test_shipping_mode = shipping_modes[0]

filtered_test_df = apply_dashboard_filters(
    df,
    {
        "Shipping Mode": [test_shipping_mode]
    }
)

print("\nSelected Shipping Mode:", test_shipping_mode)

print("Original Records:", len(df))

print("Filtered Records:", len(filtered_test_df))

print(
    "Filter Correct:",
    filtered_test_df["Shipping Mode"]
    .eq(test_shipping_mode)
    .all()
)

Available Shipping Modes:
['First Class', 'Same Day', 'Second Class', 'Standard Class']

Selected Shipping Mode: First Class
Original Records: 180519
Filtered Records: 27814
Filter Correct: True


In [6]:
# ============================================================
# 6. INDIVIDUAL FILTER TESTS
# ============================================================

filter_test_results = []

for column in FILTER_COLUMNS:

    if column not in df.columns:
        filter_test_results.append({
            "Test_ID": f"F-{column}",
            "Filter_Column": column,
            "Selected_Value": None,
            "Filtered_Records": 0,
            "Test_Status": "FAIL - Column Missing"
        })
        continue

    available_values = (
        df[column]
        .dropna()
        .unique()
        .tolist()
    )

    if not available_values:
        filter_test_results.append({
            "Test_ID": f"F-{column}",
            "Filter_Column": column,
            "Selected_Value": None,
            "Filtered_Records": 0,
            "Test_Status": "REVIEW - No Values"
        })
        continue

    selected_value = available_values[0]

    filtered_data = apply_dashboard_filters(
        df,
        {
            column: [selected_value]
        }
    )

    test_passed = (
        filtered_data[column]
        .eq(selected_value)
        .all()
    )

    filter_test_results.append({
        "Test_ID": f"F-{column}",
        "Filter_Column": column,
        "Selected_Value": str(selected_value),
        "Filtered_Records": len(filtered_data),
        "Test_Status": (
            "PASS" if test_passed else "FAIL"
        )
    })

filter_test_results_df = pd.DataFrame(
    filter_test_results
)

display(filter_test_results_df)

,Test_ID,Filter_Column,Selected_Value,Filtered_Records,Test_Status
0,F-Shipping Mode,Shipping Mode,Standard Class,107752,PASS
1,F-Delivery Status,Delivery Status,Late delivery,98977,PASS
2,F-Order Region,Order Region,South Asia,7731,PASS
3,F-Market,Market,Pacific Asia,41260,PASS
4,F-Customer Segment,Customer Segment,Consumer,93504,PASS
5,F-Order Status,Order Status,COMPLETE,59491,PASS
6,F-Category Name,Category Name,Cardio Equipment,12487,PASS
7,F-Department Name,Department Name,Footwear,14525,PASS


In [7]:
# ============================================================
# 7. COMBINED FILTER TEST
# ============================================================

combined_filters = {
    "Shipping Mode": ["First Class"],
    "Market": ["Europe"],
    "Customer Segment": ["Consumer"]
}

# Confirm that the test values exist.
for column, selected_values in combined_filters.items():

    if column not in df.columns:
        raise KeyError(
            f"Missing filter column: {column}"
        )

    available_values = (
        df[column]
        .dropna()
        .unique()
        .tolist()
    )

    invalid_values = [
        value
        for value in selected_values
        if value not in available_values
    ]

    if invalid_values:
        print(
            f"Review {column}: "
            f"values not present in dataset: "
            f"{invalid_values}"
        )

combined_filtered_df = apply_dashboard_filters(
    df,
    combined_filters
)

print("Original Records:", len(df))

print(
    "Combined Filter Records:",
    len(combined_filtered_df)
)

print("\nFiltered Sample:")
display(combined_filtered_df.head())

Original Records: 180519
Combined Filter Records: 4138

Filtered Sample:


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Delay_Severity,Delivery_Efficiency_Ratio,Revenue_Per_Unit,Profit_Margin,Discount_Level,Region_Market,Shipping_Mode_Rank,Is_Corporate_Customer,Order_Quantity_Category,Operational_Risk_Profile
112,PAYMENT,2,1,128.51,377.98,Late delivery,1,45,Fishing,Caguas,...,Minor Delay,0.5,399.98,0.321291,Low Discount,Europe - Northern Europe,2,0,Small,High Risk
366,PAYMENT,2,1,-2.69,47.25,Late delivery,1,24,Women's Apparel,Caguas,...,Minor Delay,0.5,50.00,-0.053800,Low Discount,Europe - Western Europe,2,0,Small,High Risk
367,PAYMENT,2,1,98.99,263.97,Late delivery,1,9,Cardio Equipment,Caguas,...,Minor Delay,0.5,99.99,0.330000,Medium Discount,Europe - Western Europe,2,0,Medium,High Risk
464,TRANSFER,2,1,-10.60,186.00,Shipping canceled,0,24,Women's Apparel,Oceanside,...,Minor Delay,0.5,50.00,-0.053000,Low Discount,Europe - Northern Europe,2,0,Large,Lower Risk
604,DEBIT,2,1,18.14,49.98,Late delivery,1,35,Golf Gloves,Freeport,...,Minor Delay,0.5,24.99,0.362945,No Discount,Europe - Southern Europe,2,0,Medium,High Risk


In [8]:
if not combined_filtered_df.empty:

    assert combined_filtered_df[
        "Shipping Mode"
    ].eq("First Class").all()

    assert combined_filtered_df[
        "Market"
    ].eq("Europe").all()

    assert combined_filtered_df[
        "Customer Segment"
    ].eq("Consumer").all()

    print("Combined filter validation passed.")

Combined filter validation passed.


In [9]:
# ============================================================
# 8. KPI CALCULATION FOR FILTERED DATA
# ============================================================

def calculate_delivery_kpis(data):

    actual_col = "Days for shipping (real)"
    scheduled_col = "Days for shipment (scheduled)"

    required_columns = [
        actual_col,
        scheduled_col
    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in data.columns
    ]

    if missing_columns:
        raise KeyError(
            f"Missing KPI columns: {missing_columns}"
        )

    actual_days = pd.to_numeric(
        data[actual_col],
        errors="coerce"
    )

    scheduled_days = pd.to_numeric(
        data[scheduled_col],
        errors="coerce"
    )

    valid = (
        actual_days.notna()
        & scheduled_days.notna()
        & actual_days.ge(0)
        & scheduled_days.ge(0)
    )

    valid_actual = actual_days.loc[valid]

    valid_scheduled = scheduled_days.loc[valid]

    delay_days = (
        valid_actual - valid_scheduled
    )

    shipment_count = len(data)

    valid_count = int(valid.sum())

    delayed_count = int(
        delay_days.gt(0).sum()
    )

    delay_rate = (
        delayed_count / valid_count * 100
        if valid_count > 0
        else np.nan
    )

    average_delay = (
        delay_days.loc[delay_days.gt(0)].mean()
        if delayed_count > 0
        else np.nan
    )

    return {
        "Total_Shipment_Records": shipment_count,
        "Valid_Duration_Records": valid_count,
        "Delayed_Records": delayed_count,
        "Delay_Rate_Percent": delay_rate,
        "Average_Delay_Among_Late_Records": average_delay
    }


# Calculate KPIs for the complete dataset.
overall_kpis = calculate_delivery_kpis(df)

# Calculate KPIs for the filtered dataset.
filtered_kpis = calculate_delivery_kpis(
    combined_filtered_df
)

print("Overall KPIs:")
display(
    pd.DataFrame([overall_kpis])
)

print("\nFiltered KPIs:")
display(
    pd.DataFrame([filtered_kpis])
)

Overall KPIs:


,Total_Shipment_Records,Valid_Duration_Records,Delayed_Records,Delay_Rate_Percent,Average_Delay_Among_Late_Records
0,180519,180519,103400,57.279289,1.616741



Filtered KPIs:


,Total_Shipment_Records,Valid_Duration_Records,Delayed_Records,Delay_Rate_Percent,Average_Delay_Among_Late_Records
0,4138,4138,4138,100.0,1.0


In [10]:
# ============================================================
# 9. EDGE-CASE TESTS
# ============================================================

edge_case_results = []

# E01: No filters.
no_filter_df = apply_dashboard_filters(df, {})

edge_case_results.append({
    "Test_ID": "E01",
    "Scenario": "No filters",
    "Expected": len(df),
    "Actual": len(no_filter_df),
    "Status": (
        "PASS" if len(no_filter_df) == len(df)
        else "FAIL"
    )
})

# E02: Empty selection means no restriction.
empty_selection_df = apply_dashboard_filters(
    df,
    {"Shipping Mode": []}
)

edge_case_results.append({
    "Test_ID": "E02",
    "Scenario": "Empty selection",
    "Expected": len(df),
    "Actual": len(empty_selection_df),
    "Status": (
        "PASS"
        if len(empty_selection_df) == len(df)
        else "FAIL"
    )
})

# E03: A selection that cannot match a real value.
impossible_filter_df = apply_dashboard_filters(
    df,
    {"Shipping Mode": ["__NO_SUCH_MODE__"]}
)

edge_case_results.append({
    "Test_ID": "E03",
    "Scenario": "No matching records",
    "Expected": 0,
    "Actual": len(impossible_filter_df),
    "Status": (
        "PASS"
        if impossible_filter_df.empty
        else "FAIL"
    )
})

# E04: Invalid column should raise a KeyError.
try:

    apply_dashboard_filters(
        df,
        {"__INVALID_COLUMN__": ["Value"]}
    )

    invalid_column_status = "FAIL"

except KeyError:

    invalid_column_status = "PASS"

edge_case_results.append({
    "Test_ID": "E04",
    "Scenario": "Invalid column",
    "Expected": "KeyError",
    "Actual": invalid_column_status,
    "Status": invalid_column_status
})

edge_case_results_df = pd.DataFrame(
    edge_case_results
)

display(edge_case_results_df)

,Test_ID,Scenario,Expected,Actual,Status
0,E01,No filters,180519,180519,PASS
1,E02,Empty selection,180519,180519,PASS
2,E03,No matching records,0,0,PASS
3,E04,Invalid column,KeyError,PASS,PASS


In [11]:
# ============================================================
# 10. SAVE DAY 20 TEST OUTPUTS
# ============================================================

output_files = {
    "APL_Day20_Filter_Test_Results.csv":
        filter_test_results_df,

    "APL_Day20_Edge_Case_Test_Results.csv":
        edge_case_results_df
}

for filename, output_df in output_files.items():

    output_path = ANALYSIS_DATA_DIR / filename

    output_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("Saved:", output_path)

Saved: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data\APL_Day20_Filter_Test_Results.csv
Saved: D:\Data Analytics Project\Delivery Performance, Delay Risk, and Logistics Efficiency Analysis in Global Supply Chain Operations\data\analysis_data\APL_Day20_Edge_Case_Test_Results.csv


In [12]:
# ============================================================
# 11. FINAL OUTPUT VALIDATION
# ============================================================

validation_results = []

for filename, expected_df in output_files.items():

    output_path = ANALYSIS_DATA_DIR / filename

    exists = output_path.exists()

    file_size = (
        output_path.stat().st_size
        if exists
        else 0
    )

    validation_results.append({
        "File_Name": filename,
        "Exists": exists,
        "Expected_Rows": len(expected_df),
        "Expected_Columns": len(expected_df.columns),
        "File_Size_Bytes": file_size,
        "Status": (
            "PASS"
            if exists and file_size > 0
            else "FAIL"
        )
    })

output_validation_df = pd.DataFrame(
    validation_results
)

display(output_validation_df)

assert output_validation_df["Exists"].all(), (
    "One or more Day 20 output files are missing."
)

assert (
    output_validation_df["File_Size_Bytes"] > 0
).all(), (
    "One or more Day 20 output files are empty."
)

print("DAY 20 OUTPUT FILES VALIDATED.")

,File_Name,Exists,Expected_Rows,Expected_Columns,File_Size_Bytes,Status
0,APL_Day20_Filter_Test_Results.csv,True,8,5,499,PASS
1,APL_Day20_Edge_Case_Test_Results.csv,True,4,5,192,PASS


DAY 20 OUTPUT FILES VALIDATED.
